# XAU/USD forecasting with XGBOOST

Standalone adaptation of the matching `BTC_TEST` notebook and the `time-series-forecasting` skill. Source: `CSV/XAU_USD Historical Data.csv`; target: `Price`; units: **USD (source price)**.

**Flow:** source audit → raw chronological 80/20 Train/Test (no validation) → descriptive EDA → Train-only development and diagnostics → frozen rolling-origin Test forecasts → separate latest-origin future forecasts → RMSE/MAPE by horizon → plots and execution-generated summary.

**Asset adaptation declared before Test:** horizons [1, 5, 10, 20] count successive **observations**, not calendar days or a guaranteed month. Window models use 40 observed returns (roughly eight 5-day weeks, or fewer weeks for the WTI source). No artificial weekend/holiday prices. The supplied CSV has no exchange-calendar metadata; missing dates are disclosed. Future dates are projected from a declared weekday pattern and are not confirmed exchange sessions.

Run all cells from a fresh **Python 3.12 (vs_nine TensorFlow)** kernel using `C:/Users/User/anaconda3/envs/vs_nine_tf/python.exe`. Versions and actual executable are printed and saved. Output: `OUTPUT/XGBOOST_XAU/{tables,figures,summaries,models}`.

The filename identifies XAU/USD. The CSV does not identify fixing, venue, close time, timezone, or adjustments. Gold is conventionally quoted per troy ounce; this file has no physical-unit metadata, so metrics retain source USD price units.

**Horizon update:** `[1, 3, 5, 7, 14, 21, 30]` observed steps, as explicitly requested by the user. Previous cell outputs were cleared. Restart the kernel and run all cells to regenerate models, forecasts, metrics and exported results for this configuration; existing files in OUTPUT still belong to the previous execution until rerun.


## 1. Imports and environment

In [1]:
from pathlib import Path
import os, sys, json, random, hashlib, warnings, time
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import joblib
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from IPython.display import display, Markdown, HTML

from xgboost import XGBRegressor

## 2. Single configuration cell
All experiment settings are declared before Test inference. These are reference-informed fixed settings, not claimed asset-specific optima. Change them only as a new, explicitly documented experiment after viewing Test.

In [2]:
ASSET = 'XAU'
ASSET_NAME = 'XAU/USD'
MODEL = 'XGBOOST'
UNIT = 'USD (source price)'
ASSET_NOTE = 'The filename identifies XAU/USD. The CSV does not identify fixing, venue, close time, timezone, or adjustments. Gold is conventionally quoted per troy ounce; this file has no physical-unit metadata, so metrics retain source USD price units.'
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'CSV' / 'XAU_USD Historical Data.csv').is_file()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Start Jupyter in PROJECT_2026 or the asset notebook folder.')
DATA_PATH = PROJECT_ROOT / 'CSV' / 'XAU_USD Historical Data.csv'
DATE_COL, TARGET_COL, DATE_FORMAT = 'Date', 'Price', '%m/%d/%Y'
TRAIN_RATIO = 0.8
HORIZONS = [1, 3, 5, 7, 14, 21, 30]
DAILY_FORECAST_WINDOWS = [7, 30]
LOOKBACK = 40
TRANSFORMATION = 'log price for ARIMA; observed-step log returns for feature/MIMO models'
WEEKMASK = 'Mon Tue Wed Thu Fri'
CALENDAR_NOTE = 'Observation-index modeling; source dates retained without filling gaps. Future dates use the declared weekday pattern only; holidays, omitted records and source-session conventions remain unknown.'
CALENDAR_POLICY_USER_ACKNOWLEDGED = False
SEEDS = [0]
ALPHA = 0.05
DIAGNOSTIC_LAGS = [10,20,30]
ROLLING_WINDOW = 20
SETTINGS = {'objective': 'reg:squarederror', 'n_estimators': 100, 'learning_rate': 0.3, 'max_depth': 6, 'min_child_weight': 1, 'gamma': 0, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'tree_method': 'hist', 'random_state': 0, 'n_jobs': 2}
PACKAGES = ['numpy', 'pandas', 'matplotlib', 'scipy', 'statsmodels', 'scikit-learn', 'joblib', 'xgboost']
assert max(DAILY_FORECAST_WINDOWS) <= max(HORIZONS)

# Separate PI models; original point-forecast settings are unchanged.
XGB_PI_ALPHA = 0.05
XGB_PI_QUANTILES = [XGB_PI_ALPHA/2, 1-XGB_PI_ALPHA/2]


## 3. Output helpers, provenance and diagnostics
Process checks enforce chronology, leakage prevention and frozen fitted state. Statistical diagnostics with no declared selection threshold are marked **REPORTED**. No statistical test on Test controls tuning or refitting.

In [3]:
assert HORIZONS == sorted(set(HORIZONS)) and min(HORIZONS) >= 1
MAX_H = max(HORIZONS)
OUTPUT_DIR = PROJECT_ROOT / 'OUTPUT' / f'{MODEL}_{ASSET}'
TABLE_DIR, FIGURE_DIR, SUMMARY_DIR, MODEL_DIR = [OUTPUT_DIR / k for k in ['tables','figures','summaries','models']]
for directory in [TABLE_DIR, FIGURE_DIR, SUMMARY_DIR, MODEL_DIR]: directory.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.figsize': (12, 4.5), 'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .2})
def save_table(frame, name): frame.to_csv(TABLE_DIR / name, index=False, encoding='utf-8-sig')
def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / name, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
def show_table(frame):
    display(HTML('<div style="max-height:340px;overflow:auto;border:1px solid #ddd">' +
                 frame.to_html(index=False, float_format=lambda x: f'{x:,.6f}') + '</div>'))
def write_json(path, obj): path.write_text(json.dumps(obj, indent=2, default=str, ensure_ascii=False), encoding='utf-8')
def markdown_table(frame):
    def cell(x):
        if isinstance(x,(float,np.floating)): return f'{x:.6f}'
        return str(x).replace('|','/').replace('\n',' ')
    return '\n'.join(['| '+' | '.join(map(str,frame.columns))+' |',
        '| '+' | '.join(['---']*len(frame.columns))+' |']+
        ['| '+' | '.join(cell(x) for x in row)+' |' for row in frame.itertuples(index=False,name=None)])
def warning_text(caught): return ' | '.join(f'{w.category.__name__}: {w.message}' for w in caught)
versions = pd.DataFrame([(p, version(p)) for p in PACKAGES], columns=['library','version'])
save_table(versions, 'library_versions.csv')
display(versions)
print('Python:', sys.version, '\nExecutable:', sys.executable, '\nData:', DATA_PATH, '\nOutput:', OUTPUT_DIR)
configuration = dict(asset=ASSET, model=MODEL, data=str(DATA_PATH), target=TARGET_COL, date_column=DATE_COL,
    date_format=DATE_FORMAT, unit=UNIT, asset_note=ASSET_NOTE, train_ratio=TRAIN_RATIO,
    horizons=HORIZONS, daily_forecast_windows=DAILY_FORECAST_WINDOWS, horizon_unit='observations', calendar=CALENDAR_NOTE, weekmask=WEEKMASK,
    lookback=LOOKBACK, transformation=TRANSFORMATION, seeds=SEEDS, settings=SETTINGS,
    alpha=ALPHA, validation=False, evaluation='rolling origins from final Train row; frozen fitted parameters',
    design_source='BTC_TEST implementation reference + time-series-forecasting skill; asset-specific adaptation declared before Test',
    python_executable=sys.executable, python_version=sys.version)
write_json(SUMMARY_DIR / 'configuration.json', configuration)
process_rows = []
def check(name, condition, note='', statistic=None):
    process_rows.append(dict(check=name, statistic=statistic, p_value=np.nan,
                             status='PASS' if bool(condition) else 'FAIL', interpretation=note))
    save_table(pd.DataFrame(process_rows), 'process_checks.csv')
    assert condition, f'{name}: {note}'
def residual_report(values, label, model_df=0):
    values = np.asarray(values, dtype=float)
    assert np.isfinite(values).all()
    rows = []
    lags = [lag for lag in DIAGNOSTIC_LAGS if model_df < lag < len(values)//2]
    lb = acorr_ljungbox(values, lags=lags, model_df=model_df, return_df=True)
    for lag, row in lb.iterrows():
        rows.append(dict(sample=label, check=f'Ljung-Box lag {lag}', statistic=row.lb_stat,
            p_value=row.lb_pvalue, status='REPORTED', interpretation=('Evidence of serial correlation' if row.lb_pvalue <= ALPHA else 'No rejection of residual independence') + f'; model_df={model_df}; reporting only'))
    t, p = stats.ttest_1samp(values, 0.)
    rows.append(dict(sample=label, check='Residual mean t-test', statistic=t, p_value=p, status='REPORTED',
        interpretation=f'Mean={values.mean():.8g}; descriptive, dependent errors limit the t-test'))
    lm, p, _, _ = het_arch(values, nlags=min(10, len(values)//5), ddof=model_df)
    rows.append(dict(sample=label, check='ARCH LM', statistic=lm, p_value=p, status='REPORTED',
        interpretation='Conditional variance evidence; no retuning'))
    jb, p, skew, kurt = jarque_bera(values)
    rows.append(dict(sample=label, check='Jarque-Bera', statistic=jb, p_value=p, status='REPORTED',
        interpretation=f'Normality diagnostic; skew={skew:.5g}; kurtosis={kurt:.5g}; no retuning'))
    return pd.DataFrame(rows)

,library,version
0,numpy,2.5.3
1,pandas,2.3.3
2,matplotlib,3.11.1
3,scipy,1.18.1
4,statsmodels,0.15.0
5,scikit-learn,1.9.1
6,joblib,1.6.0
7,xgboost,3.4.1


Python: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)] 
Executable: C:\Users\User\anaconda3\envs\vs_nine_tf\python.exe 
Data: C:\Users\User\Desktop\PROJECT_2026\CSV\XAU_USD Historical Data.csv 
Output: C:\Users\User\Desktop\PROJECT_2026\OUTPUT\XGBOOST_XAU


## 4. Source audit and preparation
Parse US-style dates and comma-separated prices. Stop on invalid or duplicate targets. Do not resample or silently fill/drop source observations. Auxiliary OHLC/volume columns are not features. The date-gap audit distinguishes observed records from unverified holidays or omissions.

In [4]:
raw = pd.read_csv(DATA_PATH, dtype=str, encoding='utf-8-sig')
assert {DATE_COL,TARGET_COL}.issubset(raw.columns)
dates = pd.to_datetime(raw[DATE_COL], format=DATE_FORMAT, errors='coerce')
price = pd.to_numeric(raw[TARGET_COL].str.replace(',', '', regex=False).str.strip(), errors='coerce')
invalid = dates.isna() | ~np.isfinite(price) | price.le(0)
duplicates = dates.duplicated(keep=False)
save_table(raw.loc[invalid | duplicates], 'invalid_or_duplicate_source_rows.csv')
check('Valid dates, finite strictly positive targets, unique dates', not (invalid | duplicates).any(),
      'Log representation was predetermined; stop rather than impute, drop, or offset invalid targets.')
df = pd.DataFrame({'Date': dates, 'Price': price}).sort_values('Date').set_index('Date')
n = len(df)
n_train = int(TRAIN_RATIO * n)
n_test = n - n_train
T = n_train - 1
prices = df.Price.to_numpy()
log_price = np.log(prices)
returns = np.r_[np.nan, np.diff(log_price)]
check('Chronological raw 80/20 partition', df.index.is_monotonic_increasing and df.index.is_unique and n_train == int(.8*n),
      'No shuffling, validation partition, or target imputation.')
check('Sufficient Train/Test history', n_train > LOOKBACK + MAX_H + 10 and n_test >= MAX_H,
      'Requested horizons and lookback must remain feasible without shortening.')
source_order = 'descending' if dates.is_monotonic_decreasing else ('ascending' if dates.is_monotonic_increasing else 'mixed')
calendar_offset = pd.offsets.CustomBusinessDay(weekmask=WEEKMASK)
expected_calendar = pd.date_range(df.index[0], df.index[-1], freq=calendar_offset)
missing_dates = expected_calendar.difference(df.index)
unexpected_dates = df.index.difference(expected_calendar)
save_table(pd.DataFrame({'date': missing_dates, 'classification': 'not observed; holiday/omission unknown; no imputation'}), 'calendar_missing_dates.csv')
save_table(pd.DataFrame({'date': unexpected_dates}), 'calendar_unexpected_dates.csv')
gaps = pd.DataFrame({'previous_date': df.index[:-1], 'next_date': df.index[1:],
                     'calendar_days': np.diff(df.index.values).astype('timedelta64[D]').astype(int)})
save_table(gaps, 'observation_gaps.csv')
quality = pd.DataFrame([dict(rows=n, source_order=source_order, invalid_targets=int(invalid.sum()),
    duplicate_dates=int(duplicates.sum()), nonpositive_targets=int(price.le(0).sum()),
    missing_relative_to_weekmask=len(missing_dates), unexpected_weekdays=len(unexpected_dates),
    maximum_calendar_gap=int(gaps.calendar_days.max()), volume_missing=int(raw['Vol.'].isna().sum()) if 'Vol.' in raw else None,
    first_date=df.index[0], last_date=df.index[-1], source_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    gap_policy=CALENDAR_NOTE)])
save_table(quality, 'data_quality.csv')
save_table(df.reset_index(), 'prepared_target.csv')
display(quality.T)
show_table(gaps.loc[gaps.calendar_days.gt(3)])
show_table(pd.DataFrame({'unobserved_date': missing_dates}))
print(ASSET_NOTE)
print(CALENDAR_NOTE)
print('Only Date and Price are model inputs; blank auxiliary volume is retained in source and is not imputed.')

,0
rows,1350
source_order,descending
invalid_targets,0
duplicate_dates,0
nonpositive_targets,0
missing_relative_to_weekmask,5
unexpected_weekdays,0
maximum_calendar_gap,4
volume_missing,1350
first_date,2021-07-01 00:00:00


previous_date,next_date,calendar_days
2023-04-06,2023-04-10,4
2024-03-28,2024-04-01,4
2025-04-17,2025-04-21,4


unobserved_date
2023-04-07
2024-03-29
2024-12-25
2025-01-01
2025-04-18


The filename identifies XAU/USD. The CSV does not identify fixing, venue, close time, timezone, or adjustments. Gold is conventionally quoted per troy ounce; this file has no physical-unit metadata, so metrics retain source USD price units.
Observation-index modeling; source dates retained without filling gaps. Future dates use the declared weekday pattern only; holidays, omitted records and source-session conventions remain unknown.
Only Date and Price are model inputs; blank auxiliary volume is retained in source and is not imputed.


## 5. Raw chronological Train/Test split
Split before learned scaling and windows; exact counts and dates are derived from this file. No validation subset.

In [5]:
split_summary = pd.DataFrame([dict(partition=label, rows=b-a, start_date=df.index[a], end_date=df.index[b-1])
                             for label,a,b in [('Train',0,n_train),('Test',n_train,n)]])
save_table(split_summary, 'split_summary.csv')
display(split_summary)
fig, ax = plt.subplots()
ax.plot(df.index[:n_train], prices[:n_train], label='Train 80%')
ax.plot(df.index[n_train:], prices[n_train:], label='Test 20%')
ax.axvline(df.index[n_train], color='black', linestyle='--')
ax.set(title=f'{ASSET_NAME}: raw chronological split; no validation', ylabel=UNIT, xlabel='Source date')
ax.legend(); save_figure(fig, '01_raw_split.png')

,partition,rows,start_date,end_date
0,Train,1080,2021-07-01,2025-08-27
1,Test,270,2025-08-28,2026-09-09


C:\Users\User\AppData\Local\Temp\ipykernel_13396\2788650210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Descriptive EDA
These full-sample plots describe the supplied data; model settings remain the previously declared ones.

In [6]:
# Full-sample descriptive plots do not supply model settings or selection decisions.
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0,0].plot(df.index[1:], returns[1:], linewidth=.7)
axes[0,0].set(title='Observed-step log returns (descriptive)', ylabel='Log return')
axes[0,1].hist(returns[1:], bins=50, density=True)
axes[0,1].set(title='Observed-step return distribution', xlabel='Log return')
stats.probplot(returns[1:], dist='norm', plot=axes[1,0])
axes[1,0].set_title('Normal QQ plot (descriptive)')
rolling = pd.Series(returns, index=df.index).rolling(ROLLING_WINDOW)
axes[1,1].plot(rolling.mean(), label='Rolling mean')
axes[1,1].plot(rolling.std(), label='Rolling SD')
axes[1,1].set_title(f'{ROLLING_WINDOW}-observation rolling statistics')
axes[1,1].legend(); save_figure(fig, '02_descriptive_eda.png')

C:\Users\User\AppData\Local\Temp\ipykernel_13396\2788650210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Descriptive Statistics

Price and observed-step log return, reported separately for the full sample,
the first 80% (Train), and the final 20% (Test), using the notebook's raw
chronological split before scaling or windowing. Columns identify each sample;
rows give units, dates, Count, Mean, SD, Min, Q1, Median, Q3, Max, Skewness,
and Excess kurtosis.

SD uses `ddof=1`; skewness and excess kurtosis use pandas' bias-corrected
estimators (normal-distribution excess kurtosis = 0). Log returns are natural-log
changes, not percentages, across successive observed dates. The first full-sample
return is undefined and excluded. Returns belong to their ending date, so the
first Test return uses the last Train price. Calendar gaps are not filled.
Full-sample and Test statistics are descriptive reporting only, not inputs to
model selection or fitting. The tables below were calculated from the supplied
source data without rerunning model training.


In [7]:
# Descriptive reporting only; this cell does not fit or configure a model.
def _make_descriptive_statistics(price, train_ratio):
    assert price.index.is_monotonic_increasing and price.index.is_unique
    assert price.notna().all() and np.isfinite(price).all() and price.gt(0).all()
    split = int(train_ratio * len(price))
    assert 0 < split < len(price)
    # Assign returns to their ending date; retain the Train-to-Test boundary return.
    log_return = np.log(price).diff()
    rows = []
    for variable, series, unit in [('Price', price, 'USD (source price)'),
                                    ('Log return', log_return, 'natural log; not percent')]:
        for sample, start, end in [('Full sample', 0, len(price)),
                                    ('Train', 0, split), ('Test', split, len(price))]:
            values = series.iloc[start:end].dropna()
            summary = values.describe(percentiles=[0.25, 0.5, 0.75])
            rows.append({'Variable': variable, 'Sample': sample, 'Unit': unit,
                'Start': str(values.index[0].date()), 'End': str(values.index[-1].date()),
                'Count': int(summary['count']), 'Mean': summary['mean'],
                'SD': summary['std'], 'Min': summary['min'], 'Q1': summary['25%'],
                'Median': summary['50%'], 'Q3': summary['75%'], 'Max': summary['max'],
                'Skewness': values.skew(), 'Excess kurtosis': values.kurt()})
    return pd.DataFrame(rows)

descriptive_statistics = _make_descriptive_statistics(df['Price'], TRAIN_RATIO)
del _make_descriptive_statistics
for _descriptive_variable in ['Price', 'Log return']:
    print(_descriptive_variable)
    display(descriptive_statistics.loc[
        descriptive_statistics['Variable'].eq(_descriptive_variable)
    ].drop(columns='Variable').set_index('Sample').T)
del _descriptive_variable


Price


Sample,Full sample,Train,Test
Unit,USD (source price),USD (source price),USD (source price)
Start,2021-07-01,2021-07-01,2025-08-28
End,2026-09-09,2025-08-27,2026-09-09
Count,1350,1080,270
Mean,2625.357296,2185.753407,4383.772852
SD,1002.482149,496.511872,414.187003
Min,1621.57,1621.57,3417.2
Q1,1850.2075,1817.385,4078.3575
Median,2040.835,1962.485,4353.495
Q3,3317.14,2456.67,4664.35


Log return


Sample,Full sample,Train,Test
Unit,natural log; not percent,natural log; not percent,natural log; not percent
Start,2021-07-02,2021-07-02,2025-08-28
End,2026-09-09,2025-08-27,2026-09-09
Count,1349,1079,270
Mean,0.000673,0.000601,0.000959
SD,0.011401,0.009249,0.017564
Min,-0.103497,-0.03552,-0.103497
Q1,-0.004859,-0.004543,-0.007235
Median,0.000786,0.000586,0.002639
Q3,0.006856,0.005719,0.010419


## 8. Direct horizon models: Train windows, fitting and diagnostics
Inputs are oldest to newest; every target remains inside raw Train. SVR uses the reference analytical C/epsilon rule with three other nearest Train windows (self excluded), not cross-validation. XGBoost uses fixed settings and monitors only identical fitting data. No early stopping or held-out monitoring. Training residual tests are reporting-only.

In [8]:
models, target_scalers, frozen_hashes, provenance, fit_rows, curve_rows, gain_rows = {}, {}, {}, [], [], [], []
feature_names = [f'return_lag_{lag}' for lag in range(LOOKBACK-1,-1,-1)]
training_residuals = {}
for h in HORIZONS:
    origins = np.arange(LOOKBACK, n_train-h)
    idx = origins[:,None] - np.arange(LOOKBACK-1,-1,-1)[None,:]
    targets = origins + h
    check(f'h={h}: Train target boundary', targets.max() == T and idx.min() == 1)
    check(f'h={h}: Train sample count', len(origins) == n_train-LOOKBACK-h)
    # Chronological feature ordering is identical at fit and inference for both estimators.
    X = (z if MODEL == 'SVR' else returns)[idx]
    y = log_price[targets] - log_price[origins]
    audit = pd.DataFrame(dict(origin_index=origins, origin_date=df.index[origins],
        input_start_date=df.index[idx[:,0]], input_end_date=df.index[idx[:,-1]], target_date=df.index[targets]))
    save_table(audit, f'training_window_audit_h{h}.csv')
    provenance.append(dict(h=h, n_train=len(origins), input_start_index=int(idx.min()),
        input_end_index=int(idx.max()), target_end_index=int(targets.max()), target_end_date=df.index[targets.max()]))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        if MODEL == 'SVR':
            target_scaler = StandardScaler().fit(y[:,None])
            target_scalers[h] = target_scaler
            ys = target_scaler.transform(y[:,None]).ravel()
            k = SETTINGS['knn_k']
            assert len(y)>k and X.var()>0 and ys.std()>0
            neighbors = NearestNeighbors(n_neighbors=k+1).fit(X).kneighbors(X, return_distance=False)
            others = np.stack([v[v != i][:k] for i,v in enumerate(neighbors)])
            assert others.shape == (len(y),k) and (others != np.arange(len(y))[:,None]).all()
            mse_knn = np.mean((ys-ys[others].mean(axis=1))**2)
            a = len(y)**.2*k
            sigma = np.sqrt(a/(a-1)*mse_knn)
            C = max(abs(ys.mean()+3*ys.std(ddof=0)), abs(ys.mean()-3*ys.std(ddof=0)))
            epsilon = 3*sigma*np.sqrt(np.log(len(y))/len(y))
            model = SVR(kernel=SETTINGS['kernel'], gamma=SETTINGS['gamma'], C=C, epsilon=epsilon)
            model.fit(X,ys)
            fitted = target_scaler.inverse_transform(model.predict(X)[:,None]).ravel()
            row = dict(h=h, C=C, epsilon=epsilon, sigma_hat=sigma, gamma=model._gamma,
                support_vectors=int(model.n_support_.sum()), support_vector_pct=100*model.n_support_.sum()/len(y),
                fit_status=int(model.fit_status_), scaled_train_RMSE=float(np.sqrt(np.mean((ys-model.predict(X))**2))),
                target_scaler_mean=float(target_scaler.mean_[0]), target_scaler_scale=float(target_scaler.scale_[0]),
                target_scaler_n=int(target_scaler.n_samples_seen_))
            joblib.dump(model, MODEL_DIR/f'svr_h{h}.joblib')
            joblib.dump(target_scaler, MODEL_DIR/f'target_scaler_h{h}.joblib')
            frozen_hashes[h] = joblib.hash((model,scaler,target_scaler))
            check(f'h={h}: SVR successful fit', model.fit_status_ == 0 and not any(issubclass(w.category,ConvergenceWarning) for w in caught))
        else:
            model = XGBRegressor(**SETTINGS)
            # Identical fitting observations only: validation_0 is an API label, not a validation split.
            model.fit(X,y,eval_set=[(X,y)],verbose=False)
            fitted = model.predict(X)
            losses = model.evals_result()['validation_0']['rmse']
            curve_rows.extend(dict(h=h, iteration=i+1, train_RMSE=value) for i,value in enumerate(losses))
            raw_gain = model.get_booster().get_score(importance_type='gain')
            gain_rows.extend(dict(h=h, feature=feature_names[j], lag=LOOKBACK-1-j, gain=raw_gain.get(f'f{j}',0.)) for j in range(LOOKBACK))
            row = dict(h=h, n_estimators=model.get_booster().num_boosted_rounds(), first_train_RMSE=losses[0], final_train_RMSE=losses[-1])
            check(f'h={h}: XGBoost fixed training budget', model.get_booster().num_boosted_rounds() == SETTINGS['n_estimators'] and np.isfinite(losses).all())
            model.save_model(MODEL_DIR/f'xgboost_h{h}.json')
            frozen_hashes[h] = hashlib.sha256(model.get_booster().save_raw()).hexdigest()
    row.update(n_train=len(y), warnings=warning_text(caught), status='REPORTED',
               train_RMSE_log_return=float(np.sqrt(np.mean((y-fitted)**2))))
    models[h] = model
    fit_rows.append(row)
    training_residuals[h] = y-fitted
    print(f'{MODEL} h={h}: fitted {len(y)} Train windows', flush=True)
fit_audit = pd.DataFrame(fit_rows)
save_table(fit_audit, 'training_fit.csv')
save_table(pd.DataFrame(provenance), 'training_provenance.csv')
display(fit_audit)
train_diagnostics = residual_report(training_residuals[HORIZONS[0]], f'Train cumulative log-return errors h={HORIZONS[0]}')
save_table(train_diagnostics, 'train_residual_diagnostics.csv'); display(train_diagnostics)
if MODEL == 'XGBOOST':
    curves, gains = pd.DataFrame(curve_rows), pd.DataFrame(gain_rows)
    save_table(curves,'training_curves.csv'); save_table(gains,'gain_importance.csv')
    fig,ax=plt.subplots(figsize=(10,4))
    for h in HORIZONS:
        c=curves.loc[curves.h.eq(h)]
        ax.plot(c.iteration,c.train_RMSE,label=f'h={h}')
    ax.set(title='Train curve only',xlabel='Boosting round',ylabel='RMSE (cumulative log return)')
    ax.legend(); save_figure(fig,'03_training_curve.png')
def model_fingerprint():
    if MODEL == 'SVR': return {h:joblib.hash((models[h],scaler,target_scalers[h])) for h in HORIZONS}
    return {h:hashlib.sha256(models[h].get_booster().save_raw()).hexdigest() for h in HORIZONS}
frozen_state = model_fingerprint()
assert frozen_state == frozen_hashes
def predict_cumulative(origins):
    idx = np.asarray(origins)[:,None] - np.arange(LOOKBACK-1,-1,-1)[None,:]
    assert idx.min()>=1 and (idx<=np.asarray(origins)[:,None]).all()
    X=(z if MODEL=='SVR' else returns)[idx]
    result={}
    for h in HORIZONS:
        pred=models[h].predict(X)
        result[h]=target_scalers[h].inverse_transform(pred[:,None]).ravel() if MODEL=='SVR' else pred
    return result, {}

XGBOOST h=1: fitted 1039 Train windows


XGBOOST h=3: fitted 1037 Train windows


XGBOOST h=5: fitted 1035 Train windows


XGBOOST h=7: fitted 1033 Train windows


XGBOOST h=14: fitted 1026 Train windows


XGBOOST h=21: fitted 1019 Train windows


XGBOOST h=30: fitted 1010 Train windows


,h,n_estimators,first_train_RMSE,final_train_RMSE,n_train,warnings,status,train_RMSE_log_return
0,1,100,0.008902,0.000618,1039,,REPORTED,0.000618
1,3,100,0.014661,0.000590,1037,,REPORTED,0.000590
2,5,100,0.018833,0.000639,1035,,REPORTED,0.000639
3,7,100,0.022152,0.000676,1033,,REPORTED,0.000676
4,14,100,0.030761,0.000585,1026,,REPORTED,0.000585
5,21,100,0.035804,0.000602,1019,,REPORTED,0.000602
6,30,100,0.040925,0.000596,1010,,REPORTED,0.000596


C:\Users\User\anaconda3\envs\vs_nine_tf\Lib\site-packages\statsmodels\stats\diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(


,sample,check,statistic,p_value,status,interpretation
0,Train cumulative log-return errors h=1,Ljung-Box lag 10,17.629770,6.153941e-02,REPORTED,No rejection of residual independence; model_d...
1,Train cumulative log-return errors h=1,Ljung-Box lag 20,30.762919,5.839913e-02,REPORTED,No rejection of residual independence; model_d...
2,Train cumulative log-return errors h=1,Ljung-Box lag 30,36.186347,2.020395e-01,REPORTED,No rejection of residual independence; model_d...
3,Train cumulative log-return errors h=1,Residual mean t-test,-0.000006,9.999955e-01,REPORTED,"Mean=-1.0792311e-10; descriptive, dependent er..."
4,Train cumulative log-return errors h=1,ARCH LM,115.866751,3.482215e-20,REPORTED,Conditional variance evidence; no retuning
5,Train cumulative log-return errors h=1,Jarque-Bera,1092.774723,5.093121e-238,REPORTED,Normality diagnostic; skew=-0.014164; kurtosis...


C:\Users\User\AppData\Local\Temp\ipykernel_13396\2788650210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.1 XGBoost PI 95% - Train-only quantile models

The existing squared-error point models and forecasts remain unchanged. Separate `reg:quantileerror` models estimate the 2.5% and 97.5% quantiles of cumulative log returns for horizons 1 through 30. They use the same Train windows, original fixed tree settings and budgets, with no validation, early stopping, Test tuning or all-data refit.

Columns `xgb_lo95` and `xgb_hi95` are the nominal central 95% prediction bounds in the original price units. Quantile crossings are disclosed and resolved by a fixed row-wise ordering before exponentiation. The original point forecast is not used to stretch bounds.

These are estimated quantile intervals, not conformally calibrated intervals or a guarantee of 95% coverage under time dependence or regime changes. Test coverage below is reporting only and never changes the bounds.

Method reference: [Official XGBoost quantile regression example](https://xgboost.readthedocs.io/en/release_2.0.0/python/examples/quantile_regression.html).


In [9]:
# Nominal central 95% prediction intervals from separate Train-only quantile models.
# Do not tune or calibrate these bounds using Test outcomes.
assert tuple(int(v) for v in version('xgboost').split('.')[:2]) >= (2, 0)
pi_params = dict(SETTINGS)
pi_params.update(objective='reg:quantileerror', quantile_alpha=XGB_PI_QUANTILES)
assert SETTINGS['objective'] == 'reg:squarederror'
xgb_pi_models, xgb_pi_hashes, xgb_pi_fit_rows = {}, {}, []
xgb_pi_crossing_rows = []
for h in range(1, max(DAILY_FORECAST_WINDOWS)+1):
    pi_origins = np.arange(LOOKBACK, T-h+1)
    pi_input_idx = pi_origins[:, None] - np.arange(LOOKBACK-1, -1, -1)[None, :]
    pi_target_idx = pi_origins+h
    assert pi_input_idx.min() >= 1 and pi_target_idx.max() == T
    assert (pi_input_idx <= pi_origins[:, None]).all()
    assert len(pi_origins) == n_train-LOOKBACK-h
    pi_X = returns[pi_input_idx]
    pi_y = np.asarray(log_price)[pi_target_idx]-np.asarray(log_price)[pi_origins]
    assert np.isfinite(pi_X).all() and np.isfinite(pi_y).all()
    pi_model = XGBRegressor(**pi_params)
    with warnings.catch_warnings(record=True) as pi_warnings:
        warnings.simplefilter('always')
        pi_model.fit(pi_X, pi_y, verbose=False)
    assert pi_model.get_booster().num_boosted_rounds() == SETTINGS['n_estimators']
    xgb_pi_models[h] = pi_model
    xgb_pi_hashes[h] = hashlib.sha256(pi_model.get_booster().save_raw()).hexdigest()
    xgb_pi_fit_rows.append(dict(h=h, n_train=len(pi_origins),
        input_max_index=int(pi_input_idx.max()), target_max_index=int(pi_target_idx.max()),
        target_max_date=df.index[pi_target_idx.max()], lower_quantile=XGB_PI_QUANTILES[0],
        upper_quantile=XGB_PI_QUANTILES[1], nominal_coverage=1-XGB_PI_ALPHA,
        warnings=' | '.join(str(w.message) for w in pi_warnings)))
    pi_model.save_model(MODEL_DIR/f'xgb_pi_h{h}.json')
xgb_pi_fit_audit = pd.DataFrame(xgb_pi_fit_rows)
save_table(xgb_pi_fit_audit, 'xgb_pi_training_audit.csv')
display(Markdown('**PI models: Train-only cumulative-return quantiles, fixed budgets**'))
show_table(xgb_pi_fit_audit)

def attach_xgb_pi(frame, label):
    result = frame.copy(deep=True)
    previous_columns = list(frame.columns)
    lower = np.empty(len(frame), dtype=float)
    upper = np.empty(len(frame), dtype=float)
    for h in sorted(frame['h'].unique()):
        positions = np.flatnonzero(frame['h'].to_numpy() == h)
        origins = df.index.get_indexer(pd.to_datetime(frame['origin_date'].iloc[positions]))
        assert (origins >= LOOKBACK).all()
        idx = origins[:, None] - np.arange(LOOKBACK-1, -1, -1)[None, :]
        assert idx.min() >= 1 and (idx <= origins[:, None]).all()
        X = returns[idx]
        pi_model = xgb_pi_models[int(h)]
        assert hashlib.sha256(pi_model.get_booster().save_raw()).hexdigest() == xgb_pi_hashes[int(h)]
        quantiles = pi_model.predict(X).astype(np.float64)
        assert quantiles.shape == (len(positions), 2) and np.isfinite(quantiles).all()
        crossed = quantiles[:, 0] > quantiles[:, 1]
        # Fixed monotone rearrangement; log ordering is preserved by exponentiation.
        ordered = np.sort(quantiles, axis=1)
        anchors = frame['price_origin'].iloc[positions].to_numpy(dtype=float)
        lower[positions] = anchors*np.exp(ordered[:, 0])
        upper[positions] = anchors*np.exp(ordered[:, 1])
        assert hashlib.sha256(pi_model.get_booster().save_raw()).hexdigest() == xgb_pi_hashes[int(h)]
        xgb_pi_crossing_rows.append(dict(table=label, h=int(h), n=len(positions),
            raw_quantile_crossings=int(crossed.sum()), action='Fixed row-wise quantile ordering'))
    result['xgb_lo95'] = lower
    result['xgb_hi95'] = upper
    assert np.isfinite(result[['xgb_lo95','xgb_hi95']]).all().all()
    assert (result.xgb_lo95 > 0).all() and (result.xgb_lo95 <= result.xgb_hi95).all()
    # Preserve every existing point forecast, actual, date and row exactly.
    pd.testing.assert_frame_equal(result[previous_columns], frame)
    save_table(pd.DataFrame(xgb_pi_crossing_rows), 'xgb_pi_crossing_audit.csv')
    return result


**PI models: Train-only cumulative-return quantiles, fixed budgets**

h,n_train,input_max_index,target_max_index,target_max_date,lower_quantile,upper_quantile,nominal_coverage,warnings
1,1039,1078,1079,2025-08-27,0.025000,0.975000,0.950000,
2,1038,1077,1079,2025-08-27,0.025000,0.975000,0.950000,
3,1037,1076,1079,2025-08-27,0.025000,0.975000,0.950000,
4,1036,1075,1079,2025-08-27,0.025000,0.975000,0.950000,
5,1035,1074,1079,2025-08-27,0.025000,0.975000,0.950000,
6,1034,1073,1079,2025-08-27,0.025000,0.975000,0.950000,
7,1033,1072,1079,2025-08-27,0.025000,0.975000,0.950000,
8,1032,1071,1079,2025-08-27,0.025000,0.975000,0.950000,
9,1031,1070,1079,2025-08-27,0.025000,0.975000,0.950000,
10,1030,1069,1079,2025-08-27,0.025000,0.975000,0.950000,


### 8.2 Gain importance (BTC Step 10 format)

Gain measures training split improvement, not causal influence or out-of-sample usefulness. Unused lags have zero gain. Raw gain values are shown; their magnitudes are not normalized across models with different target scales.

BTC-style reporting: heatmap across all horizons, Top 15 return lags for h=1, and the highest-gain lag for each horizon. Display lag_1 is the origin return; lag_40 is the oldest return. The display names map return_lag_0 to lag_1 through return_lag_39 to lag_40; fitted feature order stays unchanged. Horizons count observed records, not elapsed calendar days.


In [10]:
# Display lag_1 as the origin return, matching BTC; preserve oldest-to-newest model inputs.
importance_features = [f'lag_{j}' for j in range(1, LOOKBACK+1)]
importance_rows=[]
for h in HORIZONS:
    gain=models[h].get_booster().get_score(importance_type='gain')
    importance_rows.extend(dict(h=h,feature=f,gain=float(gain.get(f'f{LOOKBACK-j}',0)))
        for j,f in enumerate(importance_features,1))
feature_importance=pd.DataFrame(importance_rows)
save_table(feature_importance,'feature_importance.csv')
heat=feature_importance.pivot(index='h',columns='feature',values='gain').reindex(index=HORIZONS,columns=importance_features)
fig,ax=plt.subplots(figsize=(16,5))
im=ax.imshow(heat.to_numpy(),aspect='auto',cmap='viridis')
ax.set(yticks=np.arange(len(HORIZONS)),yticklabels=HORIZONS,
    xticks=np.arange(LOOKBACK),xticklabels=importance_features,xlabel='Return lag',ylabel='Horizon (observations)',title='Raw gain importance')
ax.tick_params(axis='x',labelrotation=90,labelsize=7)
fig.colorbar(im,ax=ax,label='Gain')
save_figure(fig,'09_feature_importance_heatmap.png')
H1=next(h for h in HORIZONS if h==1)
top=feature_importance.loc[feature_importance.h.eq(H1)].nlargest(15,'gain').sort_values('gain')
fig,ax=plt.subplots(figsize=(10,6))
ax.barh(top.feature,top.gain)
ax.set(title=f'Top 15 return lags: h={H1}',xlabel='Gain')
save_figure(fig,'10_feature_importance_h1.png')
top_by_horizon=feature_importance.loc[feature_importance.groupby('h').gain.idxmax()].reset_index(drop=True)
display(top_by_horizon)

C:\Users\User\AppData\Local\Temp\ipykernel_13396\2788650210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\User\AppData\Local\Temp\ipykernel_13396\2788650210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,h,feature,gain
0,1,lag_4,0.000161
1,3,lag_26,0.000373
2,5,lag_1,0.000727
3,7,lag_2,0.000782
4,14,lag_17,0.001164
5,21,lag_2,0.001710
6,30,lag_9,0.001845


### Step 10.1 - Lag ranking by Train Gain

**Ranking criterion:** descending XGBoost `gain`, separately for each forecast horizon from its fitted Train model. Gain is the average reduction in the training objective across splits using a feature. Rank 1 has the highest Gain; ties share a rank. Unused features have Gain = 0.

**Lag convention:** `lag_1` is the return observed at the forecast origin, `lag_2` is one observation before the origin, and `lag_40` is 39 observations before the origin. See `observations_before_origin` for explicit timing.

Separate tables show the Top 10 lags for **h=1, 3, 5, 7, 14, 21, 30 observations**, in that order. `highest_gain` flags the highest positive Gain within each horizon. Compare raw Gain within a horizon; target scales differ across horizons. The complete ranking for every horizon is still exported to `lag_ranking_by_gain.csv`.

**Interpretation:** these are importance ranks after fitting. All 40 return features remain available to the model. This does not select a subset, establish LOOKBACK=40 as optimal, or prove causality or out-of-sample usefulness. No Test scores enter the ranking.


In [11]:
# Rank all return lags from the existing Train-fitted feature importance.
lag_ranking = feature_importance[['h', 'feature', 'gain']].copy()
lag_ranking['observations_before_origin'] = lag_ranking['feature'].str.extract(r'lag_(\d+)', expand=False).astype(int) - 1
lag_ranking['rank'] = lag_ranking.groupby('h')['gain'].rank(method='min', ascending=False).astype(int)
lag_ranking['highest_gain'] = lag_ranking['rank'].eq(1) & lag_ranking['gain'].gt(0)
lag_ranking = lag_ranking.sort_values(
    ['h', 'rank', 'observations_before_origin'], kind='stable'
)[['h', 'rank', 'feature', 'observations_before_origin', 'gain', 'highest_gain']].reset_index(drop=True)
assert lag_ranking.groupby('h').size().eq(LOOKBACK).all()
assert not lag_ranking.duplicated(['h', 'feature']).any()
assert np.isfinite(lag_ranking['gain']).all() and lag_ranking['gain'].ge(0).all()
save_table(lag_ranking, 'lag_ranking_by_gain.csv')

# Show one Top 10 table per forecast horizon.
for horizon in HORIZONS:
    horizon_ranking = lag_ranking.loc[lag_ranking['h'].eq(horizon)]
    display(Markdown(f'**Top 10 lags for h={horizon} observations, ranked by Train Gain**'))
    display(horizon_ranking.head(10).style.format({'gain': '{:.9f}'}).hide(axis='index'))


**Top 10 lags for h=1 observations, ranked by Train Gain**

h,rank,feature,observations_before_origin,gain,highest_gain
1,1,lag_4,3,0.000161165,True
1,2,lag_6,5,0.000160180,False
1,3,lag_17,16,0.000150543,False
1,4,lag_2,1,0.000146701,False
1,5,lag_13,12,0.000128951,False
1,6,lag_3,2,0.000118079,False
1,7,lag_20,19,0.000115938,False
1,8,lag_26,25,0.000115514,False
1,9,lag_32,31,0.000111386,False
1,10,lag_24,23,0.000108249,False


**Top 10 lags for h=3 observations, ranked by Train Gain**

h,rank,feature,observations_before_origin,gain,highest_gain
3,1,lag_26,25,0.000372814,True
3,2,lag_7,6,0.000365553,False
3,3,lag_15,14,0.000361672,False
3,4,lag_2,1,0.000349176,False
3,5,lag_9,8,0.000320500,False
3,6,lag_3,2,0.000308284,False
3,7,lag_33,32,0.000300993,False
3,8,lag_16,15,0.000273961,False
3,9,lag_29,28,0.000255792,False
3,10,lag_32,31,0.000241154,False


**Top 10 lags for h=5 observations, ranked by Train Gain**

h,rank,feature,observations_before_origin,gain,highest_gain
5,1,lag_1,0,0.000726919,True
5,2,lag_2,1,0.000498485,False
5,3,lag_15,14,0.000453380,False
5,4,lag_23,22,0.000443914,False
5,5,lag_26,25,0.000411255,False
5,6,lag_14,13,0.000405608,False
5,7,lag_6,5,0.000401649,False
5,8,lag_34,33,0.000384541,False
5,9,lag_24,23,0.000372514,False
5,10,lag_29,28,0.000372422,False


... (4 outputs left out)

**Top 10 lags for h=21 observations, ranked by Train Gain**

h,rank,feature,observations_before_origin,gain,highest_gain
21,1,lag_2,1,0.001709613,True
21,2,lag_5,4,0.001489671,False
21,3,lag_13,12,0.001395392,False
21,4,lag_23,22,0.001355810,False
21,5,lag_4,3,0.001336997,False
21,6,lag_14,13,0.001302026,False
21,7,lag_17,16,0.001256438,False
21,8,lag_31,30,0.001251313,False
21,9,lag_18,17,0.001215129,False
21,10,lag_11,10,0.001198145,False


**Top 10 lags for h=30 observations, ranked by Train Gain**

h,rank,feature,observations_before_origin,gain,highest_gain
30,1,lag_9,8,0.001845352,True
30,2,lag_22,21,0.001740141,False
30,3,lag_10,9,0.001733764,False
30,4,lag_29,28,0.001695598,False
30,5,lag_27,26,0.001676960,False
30,6,lag_11,10,0.001663267,False
30,7,lag_6,5,0.001543501,False
30,8,lag_19,18,0.001542848,False
30,9,lag_12,11,0.001522821,False
30,10,lag_26,25,0.001519022,False


## 9. Rolling-origin Test forecast calculation
Origins start at the last Train row. At each origin, prior observed Test history may update inputs/filtering states. It never changes fitted coefficients, weights, scalers or tree structure. A horizon-h forecast does not use actuals between its origin and target. Targets are actual source dates at row origin+h.

In [12]:
test_origins=np.arange(T,n-1)
cumulative, seed_cumulative = predict_cumulative(test_origins)
def assemble_forecasts(predictions):
    rows=[]
    for h in HORIZONS:
        valid=test_origins+h<n
        origins=test_origins[valid]
        pred=np.asarray(predictions[h])[valid]
        rows.append(pd.DataFrame(dict(h=h,origin_index=origins,target_index=origins+h,
            origin_date=df.index[origins],target_date=df.index[origins+h],price_origin=prices[origins],
            actual=prices[origins+h],forecast=prices[origins]*np.exp(pred),predicted_log_return=pred)))
    return pd.concat(rows,ignore_index=True)
forecasts=assemble_forecasts(cumulative)
forecasts = attach_xgb_pi(forecasts, 'test')
seed_forecasts={seed:assemble_forecasts(pred) for seed,pred in seed_cumulative.items()}

## 10. Historical Test forecast tables
All rows appear in scrollable tables and are exported separately by horizon. ARIMA supplies model-based 95% intervals; log point forecasts are conditional medians. Other models have no calibrated intervals.

In [13]:
save_table(forecasts,'test_forecasts_all.csv')
for h in HORIZONS:
    group=forecasts.loc[forecasts.h.eq(h)].copy()
    save_table(group,f'forecasts_h{h}.csv')
    display(Markdown(f'### Historical Test forecasts: h={h} observations ({UNIT})'))
    show_table(group.drop(columns=['origin_index','target_index','predicted_log_return']))

### Historical Test forecasts: h=1 observations (USD (source price))

h,origin_date,target_date,price_origin,actual,forecast,xgb_lo95,xgb_hi95
1,2025-08-27,2025-08-28,"3,397.420000","3,417.200000","3,414.055135","3,367.699633","3,428.638068"
1,2025-08-28,2025-08-29,"3,417.200000","3,448.000000","3,422.399976","3,381.794648","3,451.303222"
1,2025-08-29,2025-09-01,"3,448.000000","3,476.470000","3,445.861803","3,409.585731","3,481.604654"
1,2025-09-01,2025-09-02,"3,476.470000","3,533.730000","3,457.426848","3,441.867266","3,516.551691"
1,2025-09-02,2025-09-03,"3,533.730000","3,559.430000","3,529.442272","3,492.633777","3,563.237418"
1,2025-09-03,2025-09-04,"3,559.430000","3,546.170000","3,556.393587","3,529.445739","3,598.268537"
1,2025-09-04,2025-09-05,"3,546.170000","3,586.810000","3,552.834862","3,523.952245","3,599.495071"
1,2025-09-05,2025-09-08,"3,586.810000","3,635.840000","3,576.902091","3,558.420497","3,623.727383"
1,2025-09-08,2025-09-09,"3,635.840000","3,626.480000","3,657.042762","3,603.242295","3,689.339222"
1,2025-09-09,2025-09-10,"3,626.480000","3,640.740000","3,609.181976","3,576.307092","3,665.478159"


### Historical Test forecasts: h=3 observations (USD (source price))

h,origin_date,target_date,price_origin,actual,forecast,xgb_lo95,xgb_hi95
3,2025-08-27,2025-09-01,"3,397.420000","3,476.470000","3,428.820772","3,357.512382","3,431.363082"
3,2025-08-28,2025-09-02,"3,417.200000","3,533.730000","3,393.622092","3,337.100873","3,487.168964"
3,2025-08-29,2025-09-03,"3,448.000000","3,559.430000","3,458.507253","3,396.806945","3,497.668924"
3,2025-09-01,2025-09-04,"3,476.470000","3,546.170000","3,446.845892","3,400.024387","3,544.577942"
3,2025-09-02,2025-09-05,"3,533.730000","3,586.810000","3,516.615526","3,485.360234","3,611.445187"
3,2025-09-03,2025-09-08,"3,559.430000","3,635.840000","3,598.980175","3,525.433334","3,642.153837"
3,2025-09-04,2025-09-09,"3,546.170000","3,626.480000","3,593.617512","3,514.855697","3,625.265500"
3,2025-09-05,2025-09-10,"3,586.810000","3,640.740000","3,628.369597","3,559.616472","3,674.822253"
3,2025-09-08,2025-09-11,"3,635.840000","3,634.090000","3,653.858382","3,578.887366","3,700.940847"
3,2025-09-09,2025-09-12,"3,626.480000","3,643.130000","3,662.110999","3,574.411886","3,692.479533"


### Historical Test forecasts: h=5 observations (USD (source price))

h,origin_date,target_date,price_origin,actual,forecast,xgb_lo95,xgb_hi95
5,2025-08-27,2025-09-03,"3,397.420000","3,559.430000","3,400.857674","3,320.655912","3,469.468774"
5,2025-08-28,2025-09-04,"3,417.200000","3,546.170000","3,418.228182","3,345.805424","3,477.624247"
5,2025-08-29,2025-09-05,"3,448.000000","3,586.810000","3,429.301258","3,394.283540","3,519.389529"
5,2025-09-01,2025-09-08,"3,476.470000","3,635.840000","3,415.416538","3,412.922338","3,552.802951"
5,2025-09-02,2025-09-09,"3,533.730000","3,626.480000","3,529.427107","3,507.441048","3,638.609972"
5,2025-09-03,2025-09-10,"3,559.430000","3,640.740000","3,591.759570","3,518.878710","3,672.847219"
5,2025-09-04,2025-09-11,"3,546.170000","3,634.090000","3,583.663337","3,477.402555","3,683.647634"
5,2025-09-05,2025-09-12,"3,586.810000","3,643.130000","3,625.000259","3,525.562410","3,681.302418"
5,2025-09-08,2025-09-15,"3,635.840000","3,679.130000","3,646.539118","3,505.922356","3,709.257568"
5,2025-09-09,2025-09-16,"3,626.480000","3,689.800000","3,642.094609","3,563.414038","3,693.715787"


... (4 outputs left out)

### Historical Test forecasts: h=21 observations (USD (source price))

h,origin_date,target_date,price_origin,actual,forecast,xgb_lo95,xgb_hi95
21,2025-08-27,2025-09-25,"3,397.420000","3,749.370000","3,458.393356","3,242.246617","3,514.069811"
21,2025-08-28,2025-09-26,"3,417.200000","3,760.240000","3,481.726546","3,261.703202","3,608.160205"
21,2025-08-29,2025-09-29,"3,448.000000","3,833.930000","3,433.553401","3,345.572263","3,633.039003"
21,2025-09-01,2025-09-30,"3,476.470000","3,858.510000","3,480.816930","3,361.216156","3,631.910600"
21,2025-09-02,2025-10-01,"3,533.730000","3,865.800000","3,609.599009","3,422.548775","3,706.182663"
21,2025-09-03,2025-10-02,"3,559.430000","3,856.530000","3,568.790860","3,449.766331","3,732.709394"
21,2025-09-04,2025-10-03,"3,546.170000","3,886.830000","3,523.887353","3,459.217519","3,709.593274"
21,2025-09-05,2025-10-06,"3,586.810000","3,961.090000","3,598.107975","3,410.787244","3,840.747526"
21,2025-09-08,2025-10-07,"3,635.840000","3,983.820000","3,664.183887","3,521.752445","3,782.657427"
21,2025-09-09,2025-10-08,"3,626.480000","4,038.900000","3,595.241704","3,505.346504","3,778.275238"


### Historical Test forecasts: h=30 observations (USD (source price))

h,origin_date,target_date,price_origin,actual,forecast,xgb_lo95,xgb_hi95
30,2025-08-27,2025-10-08,"3,397.420000","4,038.900000","3,358.869084","3,242.946292","3,671.414538"
30,2025-08-28,2025-10-09,"3,417.200000","3,976.290000","3,480.224602","3,254.853264","3,616.477651"
30,2025-08-29,2025-10-10,"3,448.000000","4,018.300000","3,458.394629","3,294.932773","3,693.388977"
30,2025-09-01,2025-10-13,"3,476.470000","4,110.700000","3,562.702420","3,294.689247","3,779.798344"
30,2025-09-02,2025-10-14,"3,533.730000","4,142.310000","3,665.633721","3,375.302115","3,714.666370"
30,2025-09-03,2025-10-15,"3,559.430000","4,208.580000","3,537.208088","3,479.147541","3,836.424233"
30,2025-09-04,2025-10-16,"3,546.170000","4,326.180000","3,603.653274","3,390.447742","3,852.584720"
30,2025-09-05,2025-10-17,"3,586.810000","4,249.980000","3,620.322522","3,616.792879","3,921.742381"
30,2025-09-08,2025-10-20,"3,635.840000","4,356.500000","3,741.135622","3,515.171839","3,899.906644"
30,2025-09-09,2025-10-21,"3,626.480000","4,125.100000","3,588.365163","3,455.519338","3,844.235120"


## 11. Future forecasts from the latest observed origin
Same original Train parameters; no all-data refit. One row per horizon. Future dates are conditional weekday-pattern projections, explicitly distinct from observed historical target dates. No actual values or evaluation metrics exist for these rows.

In [14]:
# These are projected source-calendar dates, not verified exchange/holiday sessions.
future_dates=pd.date_range(df.index[-1]+calendar_offset,periods=MAX_H,freq=calendar_offset)
future_rows=[]
if MODEL=='ARIMA':
    latest_state=res.apply(log_price,refit=False)
    np.testing.assert_array_equal(latest_state.params,frozen_params)
    fc=latest_state.get_forecast(steps=MAX_H)
    means=np.asarray(fc.predicted_mean); intervals=np.asarray(fc.conf_int(alpha=ALPHA))
    for h in HORIZONS:
        future_rows.append(dict(h=h,origin_date=df.index[-1],target_date=future_dates[h-1],
            target_date_status='PROJECTED: weekday pattern, holidays/omissions unknown',price_origin=prices[-1],
            forecast=np.exp(means[h-1]),lower95=np.exp(intervals[h-1,0]),upper95=np.exp(intervals[h-1,1])))
else:
    latest,_=predict_cumulative(np.array([n-1]))
    for h in HORIZONS:
        future_rows.append(dict(h=h,origin_date=df.index[-1],target_date=future_dates[h-1],
            target_date_status='PROJECTED: weekday pattern, holidays/omissions unknown',price_origin=prices[-1],
            forecast=prices[-1]*np.exp(latest[h][0])))
future_forecasts=pd.DataFrame(future_rows)
future_forecasts = attach_xgb_pi(future_forecasts, 'future_by_horizon')
save_table(future_forecasts,'future_forecasts.csv')
display(Markdown('### Future forecasts: common latest origin; original Train fit'))
show_table(future_forecasts)
print('Future values have no actuals or metrics. Forecasts count future observations; displayed dates are conditional projections.')
check('Fitted model and preprocessing remain frozen', model_fingerprint()==frozen_state,
      'Includes SVR scalers, XGBoost trees, LSTM weights or full PatchTST state including BatchNorm buffers.')
if MODEL in ['SVR','LSTM','TRANSFORMER']:
    check('Input scaler remains frozen',joblib.hash(scaler)==scaler_hash)
write_json(MODEL_DIR/'frozen_state_audit.json',dict(before=frozen_state,after=model_fingerprint(),
    original_fit_end_date=df.index[T],latest_observed_input_date=df.index[-1],refit=False))

### Future forecasts: common latest origin; original Train fit

h,origin_date,target_date,target_date_status,price_origin,forecast,xgb_lo95,xgb_hi95
1,2026-09-09,2026-09-10,"PROJECTED: weekday pattern, holidays/omissions unknown","4,401.550000","4,446.587060","4,361.772338","4,478.077089"
3,2026-09-09,2026-09-14,"PROJECTED: weekday pattern, holidays/omissions unknown","4,401.550000","4,411.841577","4,327.984542","4,531.533348"
5,2026-09-09,2026-09-16,"PROJECTED: weekday pattern, holidays/omissions unknown","4,401.550000","4,424.730971","4,355.106785","4,482.541426"
7,2026-09-09,2026-09-18,"PROJECTED: weekday pattern, holidays/omissions unknown","4,401.550000","4,524.366277","4,340.644675","4,573.779236"
14,2026-09-09,2026-09-29,"PROJECTED: weekday pattern, holidays/omissions unknown","4,401.550000","4,320.463039","4,323.558424","4,521.473170"
21,2026-09-09,2026-10-08,"PROJECTED: weekday pattern, holidays/omissions unknown","4,401.550000","4,314.560887","4,296.149291","4,583.723567"
30,2026-09-09,2026-10-21,"PROJECTED: weekday pattern, holidays/omissions unknown","4,401.550000","4,504.552867","4,315.807349","4,783.526225"


Future values have no actuals or metrics. Forecasts count future observations; displayed dates are conditional projections.


### 11.1 Daily future forecast tables - next 7 and 30 steps

Horizons count future observations. Dates retain the declared source-weekday pattern, including Sunday for WTI; they are conditional projections with holidays and source omissions unknown. These are not 7/30 elapsed calendar days.

Reuses original direct-horizon models; fits only missing daily horizons on Train with the original settings/prescription. No interpolation, validation, Test tuning or refitting of existing models. No calibrated prediction intervals are supplied.

Exports `tables/future_forecast_daily_7d.csv` and `tables/future_forecast_daily_30d.csv` under the existing model output directory.


In [15]:
daily_steps = max(DAILY_FORECAST_WINDOWS)
daily_origin = pd.Timestamp(df.index[-1])
daily_origin_price = float(df['Price'].iloc[-1])
daily_dates = pd.DatetimeIndex(future_dates[:daily_steps])
assert len(daily_dates) == daily_steps
assert daily_dates.is_unique and daily_dates.is_monotonic_increasing
assert (daily_dates > daily_origin).all()
future_forecast_daily = pd.DataFrame(dict(origin_date=daily_origin,
    h=np.arange(1, daily_steps + 1), target_date=daily_dates,
    day_of_week=daily_dates.day_name(), price_origin=daily_origin_price))
future_forecast_daily['target_date_status'] = 'PROJECTED: weekday pattern, holidays/omissions unknown'
# Reuse original direct-horizon models; fit only missing horizons on Train.
daily_models = {h: models[h] for h in HORIZONS}
daily_hashes = {h: frozen_hashes[h] for h in HORIZONS}
daily_fit_rows = []
for h in range(1, daily_steps + 1):
    if h in models:
        continue
    origins = np.arange(LOOKBACK, T - h + 1)
    input_idx = origins[:, None] - np.arange(LOOKBACK - 1, -1, -1)[None, :]
    target_idx = origins + h
    assert input_idx.min() >= 1 and target_idx.max() == T
    assert (input_idx <= origins[:, None]).all()
    assert len(origins) == n_train - LOOKBACK - h
    X = returns[input_idx]
    y = np.asarray(log_price)[target_idx] - np.asarray(log_price)[origins]
    assert np.isfinite(X).all() and np.isfinite(y).all()
    model = XGBRegressor(**SETTINGS)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        # eval_set is the identical Train fitting data; no held-out monitoring.
        model.fit(X, y, eval_set=[(X, y)], verbose=False)
    losses = np.asarray(model.evals_result()['validation_0']['rmse'])
    assert len(losses) == SETTINGS['n_estimators'] and np.isfinite(losses).all()
    assert model.get_booster().num_boosted_rounds() == SETTINGS['n_estimators']
    fit_details = dict(n_estimators=len(losses), first_train_RMSE=float(losses[0]),
        final_train_RMSE=float(losses[-1]))
    model.save_model(MODEL_DIR/f'xgboost_h{h}.json')
    daily_models[h] = model
    daily_hashes[h] = hashlib.sha256(model.get_booster().save_raw()).hexdigest()
    daily_fit_rows.append(dict(h=h, n_train=len(origins), input_max_idx=int(input_idx.max()),
        target_max_idx=int(target_idx.max()), target_max_date=df.index[target_idx.max()],
        warnings=' | '.join(str(w.message) for w in caught), **fit_details))
daily_additional_fit_audit = pd.DataFrame(daily_fit_rows)
save_table(daily_additional_fit_audit, 'daily_additional_fit_audit.csv')
display(Markdown('**Train-only fitting audit for additional daily horizons**'))
show_table(daily_additional_fit_audit)
daily_idx = n - 1 - np.arange(LOOKBACK - 1, -1, -1)
daily_input = returns[daily_idx].reshape(1, LOOKBACK)
daily_values = []
for h in range(1, daily_steps + 1):
    model = daily_models[h]
    assert hashlib.sha256(model.get_booster().save_raw()).hexdigest() == daily_hashes[h]
    prediction = model.predict(daily_input)
    cumulative_return = prediction[0]
    daily_values.append(np.float64(daily_origin_price)*np.exp(cumulative_return))
    assert hashlib.sha256(model.get_booster().save_raw()).hexdigest() == daily_hashes[h]
future_forecast_daily['forecast'] = daily_values
future_forecast_daily = attach_xgb_pi(future_forecast_daily, 'future_daily')
daily_forecast_columns = ['forecast']
assert len(future_forecast_daily) == daily_steps
assert future_forecast_daily['origin_date'].nunique() == 1
assert future_forecast_daily['h'].tolist() == list(range(1, daily_steps + 1))
assert future_forecast_daily['target_date'].equals(pd.Series(daily_dates, name='target_date'))
values = future_forecast_daily[daily_forecast_columns].to_numpy()
assert np.isfinite(values).all() and (values > 0).all()
np.testing.assert_allclose(future_forecast_daily.set_index('h').loc[HORIZONS, ['forecast']],
    future_forecasts.set_index('h').loc[HORIZONS, ['forecast']], rtol=1e-12)
assert (future_forecast_daily.set_index('h').loc[HORIZONS, 'target_date'].to_numpy() ==
        future_forecasts.set_index('h').loc[HORIZONS, 'target_date'].to_numpy()).all()
future_daily_tables = {}
for days in DAILY_FORECAST_WINDOWS:
    table = future_forecast_daily.iloc[:days].copy()
    assert len(table) == days and table['h'].tolist() == list(range(1, days + 1))
    future_daily_tables[days] = table
    save_table(table, f'future_forecast_daily_{days}d.csv')
    display(Markdown(f'**Next {days} projected observations - origin {daily_origin.date()}, last observed price {daily_origin_price:,.6f} {UNIT}**'))
    shown = table.copy()
    for column in ['origin_date', 'target_date']:
        shown[column] = shown[column].dt.strftime('%Y-%m-%d')
    display(HTML(shown.to_html(index=False, float_format=lambda value: f'{value:,.6f}')))
assert all((TABLE_DIR/f'future_forecast_daily_{days}d.csv').is_file() for days in DAILY_FORECAST_WINDOWS)


**Train-only fitting audit for additional daily horizons**

h,n_train,input_max_idx,target_max_idx,target_max_date,warnings,n_estimators,first_train_RMSE,final_train_RMSE
2,1038,1077,1079,2025-08-27,,100,0.012189,0.000596
4,1036,1075,1079,2025-08-27,,100,0.017104,0.000623
6,1034,1073,1079,2025-08-27,,100,0.020178,0.000586
8,1032,1071,1079,2025-08-27,,100,0.023763,0.000593
9,1031,1070,1079,2025-08-27,,100,0.025288,0.000609
10,1030,1069,1079,2025-08-27,,100,0.026848,0.000592
11,1029,1068,1079,2025-08-27,,100,0.027151,0.000677
12,1028,1067,1079,2025-08-27,,100,0.028770,0.000619
13,1027,1066,1079,2025-08-27,,100,0.029383,0.000649
15,1025,1064,1079,2025-08-27,,100,0.031751,0.000620


**Next 7 projected observations - origin 2026-09-09, last observed price 4,401.550000 USD (source price)**

origin_date,h,target_date,day_of_week,price_origin,target_date_status,forecast,xgb_lo95,xgb_hi95
2026-09-09,1,2026-09-10,Thursday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,446.587060","4,361.772338","4,478.077089"
2026-09-09,2,2026-09-11,Friday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,392.465509","4,335.673800","4,458.387823"
2026-09-09,3,2026-09-14,Monday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,411.841577","4,327.984542","4,531.533348"
2026-09-09,4,2026-09-15,Tuesday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,443.270920","4,345.535450","4,501.290521"
2026-09-09,5,2026-09-16,Wednesday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,424.730971","4,355.106785","4,482.541426"
2026-09-09,6,2026-09-17,Thursday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,379.027797","4,320.115088","4,507.188751"
2026-09-09,7,2026-09-18,Friday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,524.366277","4,340.644675","4,573.779236"


**Next 30 projected observations - origin 2026-09-09, last observed price 4,401.550000 USD (source price)**

origin_date,h,target_date,day_of_week,price_origin,target_date_status,forecast,xgb_lo95,xgb_hi95
2026-09-09,1,2026-09-10,Thursday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,446.587060","4,361.772338","4,478.077089"
2026-09-09,2,2026-09-11,Friday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,392.465509","4,335.673800","4,458.387823"
2026-09-09,3,2026-09-14,Monday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,411.841577","4,327.984542","4,531.533348"
2026-09-09,4,2026-09-15,Tuesday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,443.270920","4,345.535450","4,501.290521"
2026-09-09,5,2026-09-16,Wednesday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,424.730971","4,355.106785","4,482.541426"
2026-09-09,6,2026-09-17,Thursday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,379.027797","4,320.115088","4,507.188751"
2026-09-09,7,2026-09-18,Friday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,524.366277","4,340.644675","4,573.779236"
2026-09-09,8,2026-09-21,Monday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,432.389574","4,306.304481","4,557.212520"
2026-09-09,9,2026-09-22,Tuesday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,437.025874","4,304.239146","4,578.866807"
2026-09-09,10,2026-09-23,Wednesday,"4,401.550000","PROJECTED: weekday pattern, holidays/omissions unknown","4,375.808728","4,312.907958","4,530.460881"


### 11.2 Future forecast plots from the latest observed date

These 7- and 30-step plots reuse the future forecast tables above and the original Train-fitted model. The dashed line marks the last observed date; every forecast target lies after it. The left panel provides recent price history and the right panel enlarges the future path.

**Step unit:** projected observations. Weekday-based dates; holidays and source omissions are unknown.

**Uncertainty:** Nominal quantile interval; coverage is not guaranteed.

**Rendering note:** Added outputs were rendered from saved forecasts without retraining. On Run All, this cell uses the freshly computed in-memory forecast table. The forecast origin is the dataset end date, not the current wall-clock date.

In [16]:
# Plot the already computed future path; no fitting or new inference.
FUTURE_MODEL_LABEL = 'XGBoost'
FUTURE_UNCERTAINTY_NOTE = 'Nominal quantile interval; coverage is not guaranteed.'
FUTURE_PLOT_LABEL = 'XAU/USD'
FUTURE_PLOT_UNIT = 'USD (source price)'
FUTURE_STEP_LABEL = 'projected observations'
FUTURE_DATE_NOTE = 'Weekday-based dates; holidays and source omissions are unknown.'

def plot_model_future_path(history, future_path, steps):
    path = future_path.sort_values('h').iloc[:steps].copy()
    origin = pd.Timestamp(history.index[-1])
    last_actual = float(history.iloc[-1])
    forecast_column = next(c for c in ['svr_fc', 'xgb_fc', 'patchtst_fc_ens', 'forecast'] if c in path)
    seed_columns = [column for column in path if column.startswith(('patchtst_fc_seed', 'forecast_seed'))]
    dates = pd.DatetimeIndex(pd.to_datetime(path['target_date']))
    values = path[forecast_column].to_numpy(dtype=float)
    assert len(path) == steps and path['h'].tolist() == list(range(1, steps + 1))
    assert dates.is_unique and dates.is_monotonic_increasing and (dates > origin).all()
    assert (pd.to_datetime(path['origin_date']) == origin).all()
    assert np.isfinite(values).all() and (values > 0).all()
    np.testing.assert_allclose(path['price_origin'], last_actual, rtol=1e-12)

    joined_dates = pd.DatetimeIndex([origin, *dates])
    joined_forecast = np.r_[last_actual, values]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1.2, 1]})
    recent = history.iloc[-60:]
    axes[0].plot(recent.index, recent, color='#263648', linewidth=1.7, label='Observed price (last 60 records)')
    for ax in axes:
        ax.plot(joined_dates, joined_forecast, color='#d56b27', linewidth=2.1,
                marker='o', markersize=3, label=f'{FUTURE_MODEL_LABEL} forecast')
        ax.axvline(origin, color='#627080', linestyle='--', linewidth=1,
                   label=f'Last observed: {origin:%Y-%m-%d}')
        ax.scatter([origin], [last_actual], color='#263648', s=35, zorder=4)
        if seed_columns:
            seed_values = path[seed_columns].to_numpy(dtype=float)
            assert np.isfinite(seed_values).all() and (seed_values > 0).all()
            low = np.r_[last_actual, seed_values.min(axis=1)]
            high = np.r_[last_actual, seed_values.max(axis=1)]
            ax.fill_between(joined_dates, low, high, color='#d56b27', alpha=.13,
                            label=f'Range across {len(seed_columns)} seeds')
        if {'xgb_lo95', 'xgb_hi95'}.issubset(path.columns):
            interval = path[['xgb_lo95', 'xgb_hi95']].to_numpy(dtype=float)
            assert np.isfinite(interval).all() and (interval > 0).all()
            assert (interval[:, 0] <= interval[:, 1]).all()
            ax.fill_between(joined_dates, np.r_[last_actual, interval[:, 0]],
                            np.r_[last_actual, interval[:, 1]], color='#d56b27', alpha=.13,
                            label='Nominal 95% quantile prediction interval')
        ax.set(ylabel=FUTURE_PLOT_UNIT, xlabel='Target date')
        ax.grid(alpha=.2)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(axis='x', labelrotation=30)
    axes[0].set_title('Recent history and future forecast', loc='left', fontsize=11)
    axes[1].set_title('Future path detail', loc='left', fontsize=11)
    endpoint_label = f'{values[-1]:,.6f}' if FUTURE_PLOT_LABEL == 'USD/EUR' else f'{values[-1]:,.2f}'
    axes[1].annotate(endpoint_label, xy=(dates[-1], values[-1]),
                     xytext=(-5, 10), textcoords='offset points', ha='right', color='#a34a15')
    axes[0].legend(loc='best', fontsize=8, frameon=False)
    axes[1].legend(loc='best', fontsize=8, frameon=False)
    fig.suptitle(f'{FUTURE_PLOT_LABEL} | {FUTURE_MODEL_LABEL} | next {steps} {FUTURE_STEP_LABEL}',
                 fontsize=14, fontweight='bold', x=.055, ha='left')
    fig.text(.055, .045, f'Forecast origin: {origin:%Y-%m-%d}; last observed price: {last_actual:,.4f}. '
             'Original Train fit; no refitting.', fontsize=9, color='#405164')
    fig.text(.055, .012, FUTURE_DATE_NOTE + ' ' + FUTURE_UNCERTAINTY_NOTE,
             fontsize=8, color='#596574')
    fig.tight_layout(rect=(0, .08, 1, .92))
    return fig

for future_steps in [7, 30]:
    future_fig = plot_model_future_path(df['Price'], future_forecast_daily, future_steps)
    plt.show()
    plt.close(future_fig)


C:\Users\User\AppData\Local\Temp\ipykernel_13396\937822395.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\User\AppData\Local\Temp\ipykernel_13396\937822395.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 11.2 PI coverage on historical Test - reporting only

Nominal coverage is 95%; realized coverage and interval width are descriptive. Overlapping horizons, time dependence and regime changes limit interpretation. No settings are revised from this table.


In [17]:
xgb_pi_coverage_rows = []
for h, group in forecasts.groupby('h', sort=True):
    covered = group['actual'].between(group['xgb_lo95'], group['xgb_hi95'])
    xgb_pi_coverage_rows.append(dict(h=int(h), n=len(group), nominal_coverage_pct=95.,
        empirical_test_coverage_pct=float(100*covered.mean()),
        mean_interval_width=float((group['xgb_hi95']-group['xgb_lo95']).mean()),
        status='REPORTED', interpretation='Test reporting only; no recalibration or retuning'))
xgb_pi_test_coverage = pd.DataFrame(xgb_pi_coverage_rows)
save_table(xgb_pi_test_coverage, 'xgb_pi_test_coverage.csv')
display(xgb_pi_test_coverage)
display(Markdown('**Raw quantile crossing audit (ordering fixed before Test)**'))
display(pd.DataFrame(xgb_pi_crossing_rows))
assert all(hashlib.sha256(model.get_booster().save_raw()).hexdigest() == xgb_pi_hashes[h]
           for h, model in xgb_pi_models.items())


,h,n,nominal_coverage_pct,empirical_test_coverage_pct,mean_interval_width,status,interpretation
0,1,270,95.0,57.407407,97.896393,REPORTED,Test reporting only; no recalibration or retuning
1,3,268,95.0,49.626866,155.888200,REPORTED,Test reporting only; no recalibration or retuning
2,5,266,95.0,41.729323,186.396827,REPORTED,Test reporting only; no recalibration or retuning
3,7,264,95.0,42.045455,220.380444,REPORTED,Test reporting only; no recalibration or retuning
4,14,257,95.0,39.688716,293.377043,REPORTED,Test reporting only; no recalibration or retuning
5,21,250,95.0,35.600000,338.009752,REPORTED,Test reporting only; no recalibration or retuning
6,30,241,95.0,23.236515,382.247647,REPORTED,Test reporting only; no recalibration or retuning


**Raw quantile crossing audit (ordering fixed before Test)**

,table,h,n,raw_quantile_crossings,action
0,test,1,270,0,Fixed row-wise quantile ordering
1,test,3,268,0,Fixed row-wise quantile ordering
2,test,5,266,0,Fixed row-wise quantile ordering
3,test,7,264,0,Fixed row-wise quantile ordering
4,test,14,257,0,Fixed row-wise quantile ordering
5,test,21,250,0,Fixed row-wise quantile ordering
6,test,30,241,0,Fixed row-wise quantile ordering
7,future_by_horizon,1,1,0,Fixed row-wise quantile ordering
8,future_by_horizon,3,1,0,Fixed row-wise quantile ordering
9,future_by_horizon,5,1,0,Fixed row-wise quantile ordering


## 12. Process checks and RMSE/MAPE
Check all alignment, counts, unique origins, finite outputs and valid inverse transforms before scoring. Export offending rows and stop on failure; do not clip. Report RMSE in original units, MAPE in percent and sample size separately by horizon. Zero actuals make group MAPE undefined rather than changing the denominator.

In [18]:
def validate_forecasts(frame,label):
    cols=[c for c in ['actual','forecast','price_origin','lower95','upper95'] if c in frame]
    good=np.isfinite(frame[cols]).all(axis=1) & frame.forecast.gt(0)
    if 'actual' in frame: good &= frame.actual.gt(0)
    if 'lower95' in frame: good &= frame.lower95.gt(0) & frame.lower95.le(frame.forecast) & frame.forecast.le(frame.upper95)
    save_table(frame.loc[~good],f'{label}_offending_rows.csv')
    check(f'{label}: finite and valid log inversion',bool(good.all()),'No clipping or post-Test parameter revision.')
    check(f'{label}: unique origin/horizon',not frame.duplicated(['origin_date','h']).any())
    if 'target_index' in frame:
        check(f'{label}: target alignment',bool((frame.target_index==frame.origin_index+frame.h).all()) and
            (frame.target_index>=n_train).all() and (frame.target_index<n).all() and
            np.array_equal(pd.to_datetime(frame.target_date).to_numpy(),df.index[frame.target_index].to_numpy()) and
            np.array_equal(pd.to_datetime(frame.origin_date).to_numpy(),df.index[frame.origin_index].to_numpy()) and
            np.array_equal(frame.actual.to_numpy(),prices[frame.target_index]),
            'Dates come from actual observed row positions, never calendar-day addition.')
        for h in HORIZONS:
            check(f'{label}: h={h} count',int(frame.h.eq(h).sum())==n_test-h+1,statistic=int(frame.h.eq(h).sum()))
    else:
        check(f'{label}: future-only dates',bool((frame.target_date>df.index[-1]).all()) and len(frame)==len(HORIZONS))
validate_forecasts(forecasts,'test')
validate_forecasts(future_forecasts,'future')
def score(frame):
    rows=[]
    for h,g in frame.groupby('h',sort=True):
        actual=g.actual.to_numpy(); forecast=g.forecast.to_numpy()
        zero=int((actual==0).sum())
        rows.append(dict(h=h,n=len(g),first_target_date=g.target_date.min(),last_target_date=g.target_date.max(),
            RMSE=np.sqrt(np.mean((actual-forecast)**2)),MAPE=np.nan if zero else 100*np.mean(np.abs((actual-forecast)/actual)),
            zero_actuals=zero,RMSE_unit=UNIT,MAPE_unit='%'))
    return pd.DataFrame(rows)
evaluation=score(forecasts)
save_table(evaluation,'metrics_by_horizon.csv')
display(Markdown('### Historical Test RMSE and MAPE (after forecast tables)'))
display(evaluation)
seed_metrics=None
if seed_forecasts:
    frames=[]
    for seed,frame in seed_forecasts.items():
        validate_forecasts(frame,f'seed{seed}_test')
        save_table(frame,f'seed{seed}_test_forecasts.csv')
        metrics=score(frame); metrics.insert(0,'seed',seed); frames.append(metrics)
    seed_metrics=pd.concat(frames,ignore_index=True)
    save_table(seed_metrics,'seed_metrics.csv')
    seed_summary=seed_metrics.groupby('h').agg(RMSE_mean=('RMSE','mean'),RMSE_SD=('RMSE','std'),MAPE_mean=('MAPE','mean'),MAPE_SD=('MAPE','std')).reset_index()
    save_table(seed_summary,'seed_metric_summary.csv')
    display(Markdown('Individual seeds and sample SD (ddof=1); separate from return-space ensemble metrics'))
    display(seed_metrics); display(seed_summary)

### Historical Test RMSE and MAPE (after forecast tables)

,h,n,first_target_date,last_target_date,RMSE,MAPE,zero_actuals,RMSE_unit,MAPE_unit
0,1,270,2025-08-28,2026-09-09,87.456154,1.355551,0,USD (source price),%
1,3,268,2025-09-01,2026-09-09,144.214286,2.329731,0,USD (source price),%
2,5,266,2025-09-03,2026-09-09,178.030996,3.084408,0,USD (source price),%
3,7,264,2025-09-05,2026-09-09,215.056918,3.683490,0,USD (source price),%
4,14,257,2025-09-16,2026-09-09,289.642303,5.068154,0,USD (source price),%
5,21,250,2025-09-25,2026-09-09,335.140577,6.082083,0,USD (source price),%
6,30,241,2025-10-08,2026-09-09,403.155075,7.741811,0,USD (source price),%


## 13. Forecast plots and reporting-only Test residual diagnostics
Higher horizons can have overlapping errors and different target counts. Descriptive diagnostic p-values and Train/Test scale comparisons do not trigger model changes.

In [19]:
for h in HORIZONS:
    g=forecasts.loc[forecasts.h.eq(h)].sort_values('target_date')
    fig,ax=plt.subplots()
    ax.plot(g.target_date,g.actual,label='Actual',color='black',linewidth=1)
    ax.plot(g.target_date,g.forecast,label='Forecast',linewidth=1)
    if 'lower95' in g: ax.fill_between(g.target_date,g.lower95,g.upper95,alpha=.2,label='Model-based 95% interval')
    ax.set(title=f'{ASSET_NAME} {MODEL}: h={h} observations',xlabel='Observed target date',ylabel=UNIT)
    ax.legend(); save_figure(fig,f'test_forecast_h{h}.png')
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,metric,unit in zip(axes,['RMSE','MAPE'],[UNIT,'%']):
    ax.plot(evaluation.h,evaluation[metric],marker='o')
    ax.set(title=f'{MODEL} Test {metric}',xlabel='Horizon (observations)',ylabel=unit,xticks=HORIZONS)
save_figure(fig,'metrics_by_horizon.png')
hdiag=HORIZONS[0]
g=forecasts.loc[forecasts.h.eq(hdiag)].sort_values('origin_date')
errors=np.log(g.actual/g.price_origin)-g.predicted_log_return
test_diagnostics=residual_report(errors,f'Test cumulative log-return errors h={hdiag}; reporting only')
save_table(test_diagnostics,'test_residual_diagnostics.csv'); display(test_diagnostics)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(g.target_date,errors); axes[0].set(title=f'Test h={hdiag} log-return residuals',ylabel='Actual minus predicted log return')
plot_acf(errors,lags=min(30,len(errors)//4),ax=axes[1]); axes[1].set_title('Test residual ACF; no selection')
save_figure(fig,'test_residual_diagnostics.png')
scale_comparison=pd.DataFrame([dict(sample='Train',n=n_train-1,mean=np.mean(returns[1:n_train]),SD=np.std(returns[1:n_train],ddof=1),status='REPORTED'),
    dict(sample='Test',n=n_test,mean=np.mean(returns[n_train:]),SD=np.std(returns[n_train:],ddof=1),status='REPORTED')])
save_table(scale_comparison,'return_scale_comparison.csv'); display(scale_comparison)

C:\Users\User\AppData\Local\Temp\ipykernel_13396\2788650210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\User\anaconda3\envs\vs_nine_tf\Lib\site-packages\statsmodels\stats\diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(


,sample,check,statistic,p_value,status,interpretation
0,Test cumulative log-return errors h=1; reporti...,Ljung-Box lag 10,12.047767,2.818724e-01,REPORTED,No rejection of residual independence; model_d...
1,Test cumulative log-return errors h=1; reporti...,Ljung-Box lag 20,23.536253,2.632363e-01,REPORTED,No rejection of residual independence; model_d...
2,Test cumulative log-return errors h=1; reporti...,Ljung-Box lag 30,26.635664,6.423306e-01,REPORTED,No rejection of residual independence; model_d...
3,Test cumulative log-return errors h=1; reporti...,Residual mean t-test,0.311275,7.558325e-01,REPORTED,"Mean=0.000358674; descriptive, dependent error..."
4,Test cumulative log-return errors h=1; reporti...,ARCH LM,28.208908,1.671285e-03,REPORTED,Conditional variance evidence; no retuning
5,Test cumulative log-return errors h=1; reporti...,Jarque-Bera,228.521812,2.382973e-50,REPORTED,Normality diagnostic; skew=-0.80794; kurtosis=...


C:\Users\User\AppData\Local\Temp\ipykernel_13396\2788650210.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,sample,n,mean,SD,status
0,Train,1079,0.000601,0.009249,REPORTED
1,Test,270,0.000959,0.017564,REPORTED


## 14. Summary generated from this execution
Rerunning regenerates this report from current results; there is no copied saved-result markdown.

In [20]:
lines=[f'# {ASSET_NAME} - {MODEL}: results from this execution','',
    f'Source: `{DATA_PATH.name}`; target: {TARGET_COL}; units: {UNIT}.',ASSET_NOTE,'',
    f'Raw chronological Train/Test: {n_train}/{n_test}; no validation.',
    f'Train: {df.index[0].date()} to {df.index[T].date()}; Test: {df.index[n_train].date()} to {df.index[-1].date()}.',
    f'Horizons: {HORIZONS} observations. Lookback: {LOOKBACK} observed returns for window models.',CALENDAR_NOTE,
    f'Source calendar audit: {len(missing_dates)} dates absent relative to the declared weekday pattern; maximum gap {int(gaps.calendar_days.max())} calendar days.',
    'Missing source dates are disclosed, not filled. Observation time therefore need not equal elapsed calendar time.',
    'Rolling origins begin at the final Train observation. Each forecast uses only history through its own origin; fitted parameters remain fixed.','',
    '## Model development']
if MODEL=='ARIMA':
    lines += [f'Selected {selected.model}; Train AIC={selected.AIC:.6f}; BIC={selected.BIC:.6f}.',
        'Differencing is the smallest Train ADF/KPSS-supported order. Selection uses AIC among converged fits with valid roots. Residual tests are reported without retuning.',
        'Exponentiated log forecasts are conditional medians, not bias-corrected arithmetic means. The 95% intervals are model-based; ARCH/non-normality and calendar irregularity limit interpretation.']
    if int(selected.p)==0 and d==1 and int(selected.q)==0 and selected.trend=='n':
        lines.append('ARIMA(0,1,0) without drift gives the latest observed price as the point forecast; uncertainty increases with horizon.')
elif MODEL=='SVR':
    lines += ['Direct cumulative-log-return RBF models use the fixed Train-only C/epsilon heuristic and Train input/target scalers. No hyperparameter cross-validation.']
elif MODEL=='XGBOOST':
    lines += ['Direct cumulative-log-return models use a fixed predeclared configuration. Learning curves are fitting data only. Raw gain is neither causal importance nor proof of future usefulness.']
    lines += ['', '## Most important training lag by gain', markdown_table(top_by_horizon),
        'Display lag_1 is the origin return; lag_40 is the oldest input. Gain is Train-only and unnormalized.']
else:
    lines += [f'MIMO predicts the next {MAX_H} individual observed-step log returns; prices compound the predicted path only.',
        f'All predefined seeds {SEEDS} are included equally; {EPOCHS} fixed epochs each. Average inverse-scaled returns before compounding.',
        'Ensemble metrics are not the mean seed metrics. Seed metric SD uses ddof=1 and is not a prediction interval.']
lines += ['', '## Historical Test metrics',markdown_table(evaluation),'',
    'Horizon groups have different counts and target dates; longer-horizon forecast errors overlap. These values do not establish a monotonic relationship with horizon.',
    'MAPE is an error percentage, not accuracy. Actual targets are checked for zeros; any zero would make the full-group MAPE undefined.',
    '', '## Future forecasts (no actuals or metrics)',markdown_table(future_forecasts),'',
    'Future dates are projections from a declared source-weekday pattern, not confirmed exchange sessions. Holidays or missing future records can shift them.',
    '', '## Checks and limitations',
    f'Process checks: {sum(row["status"]=="PASS" for row in process_rows)} passed; {sum(row["status"]=="FAIL" for row in process_rows)} failed.',
    'Train/Test diagnostics with no predeclared acceptance gate are REPORTED. Passing diagnostics does not establish forecast accuracy.',
    'No separate validation estimate or validation gap exists. Train fitting loss is optimistic; fixed budgets do not establish absence of overfitting.',
    'Test findings did not change parameters, epochs, transformations, seeds, or model selection.',
    'These univariate models do not observe news, monetary policy, interest rates, inflation, or unexpected market events.',
    f'All rows, figures, summaries and fitted artifacts are under OUTPUT/{MODEL}_{ASSET}/.',
    f'Tested Python executable: {sys.executable}.']
lines += ['', '**Daily future forecasts**', 'Tables for 7 and 30 future steps are shown before metrics; dates follow the existing asset calendar convention.']
lines += ['', '**XGBoost prediction intervals**', 'xgb_lo95/xgb_hi95: nominal central 95% Train-only quantile intervals; no Test calibration or tuning. Fixed quantile rearrangement is audited. Empirical Test coverage is reported separately; 95% coverage is not guaranteed.']
summary='\n\n'.join(lines)
(SUMMARY_DIR/'execution_summary.md').write_text(summary,encoding='utf-8')
display(Markdown(summary))
write_json(SUMMARY_DIR/'execution_audit.json',dict(completed=True,asset=ASSET,model=MODEL,
    n=n,n_train=n_train,n_test=n_test,horizons=HORIZONS,counts={int(h):int(c) for h,c in forecasts.groupby('h').size().items()},
    train_end=str(df.index[T].date()),test_start=str(df.index[n_train].date()),last_observation=str(df.index[-1].date()),
    parameters_frozen=model_fingerprint()==frozen_state,process_checks=process_rows,
    source_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),seed_count=len(seed_forecasts),python_executable=sys.executable))

# XAU/USD - XGBOOST: results from this execution



Source: `XAU_USD Historical Data.csv`; target: Price; units: USD (source price).

The filename identifies XAU/USD. The CSV does not identify fixing, venue, close time, timezone, or adjustments. Gold is conventionally quoted per troy ounce; this file has no physical-unit metadata, so metrics retain source USD price units.



Raw chronological Train/Test: 1080/270; no validation.

Train: 2021-07-01 to 2025-08-27; Test: 2025-08-28 to 2026-09-09.

Horizons: [1, 3, 5, 7, 14, 21, 30] observations. Lookback: 40 observed returns for window models.

Observation-index modeling; source dates retained without filling gaps. Future dates use the declared weekday pattern only; holidays, omitted records and source-session conventions remain unknown.

Source calendar audit: 5 dates absent relative to the declared weekday pattern; maximum gap 4 calendar days.

Missing source dates are disclosed, not filled. Observation time therefore need not equal elapsed calendar time.

Rolling origins begin at the final Train observation. Each forecast uses only history through its own origin; fitted parameters remain fixed.



## Model development

Direct cumulative-log-return models use a fixed predeclared configuration. Learning curves are fitting data only. Raw gain is neither causal importance nor proof of future usefulness.



## Most important training lag by gain

| h | feature | gain |
| --- | --- | --- |
| 1 | lag_4 | 0.000161 |
| 3 | lag_26 | 0.000373 |
| 5 | lag_1 | 0.000727 |
| 7 | lag_2 | 0.000782 |
| 14 | lag_17 | 0.001164 |
| 21 | lag_2 | 0.001710 |
| 30 | lag_9 | 0.001845 |

Display lag_1 is the origin return; lag_40 is the oldest input. Gain is Train-only and unnormalized.



## Historical Test metrics

| h | n | first_target_date | last_target_date | RMSE | MAPE | zero_actuals | RMSE_unit | MAPE_unit |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 270 | 2025-08-28 00:00:00 | 2026-09-09 00:00:00 | 87.456154 | 1.355551 | 0 | USD (source price) | % |
| 3 | 268 | 2025-09-01 00:00:00 | 2026-09-09 00:00:00 | 144.214286 | 2.329731 | 0 | USD (source price) | % |
| 5 | 266 | 2025-09-03 00:00:00 | 2026-09-09 00:00:00 | 178.030996 | 3.084408 | 0 | USD (source price) | % |
| 7 | 264 | 2025-09-05 00:00:00 | 2026-09-09 00:00:00 | 215.056918 | 3.683490 | 0 | USD (source price) | % |
| 14 | 257 | 2025-09-16 00:00:00 | 2026-09-09 00:00:00 | 289.642303 | 5.068154 | 0 | USD (source price) | % |
| 21 | 250 | 2025-09-25 00:00:00 | 2026-09-09 00:00:00 | 335.140577 | 6.082083 | 0 | USD (source price) | % |
| 30 | 241 | 2025-10-08 00:00:00 | 2026-09-09 00:00:00 | 403.155075 | 7.741811 | 0 | USD (source price) | % |



Horizon groups have different counts and target dates; longer-horizon forecast errors overlap. These values do not establish a monotonic relationship with horizon.

MAPE is an error percentage, not accuracy. Actual targets are checked for zeros; any zero would make the full-group MAPE undefined.



## Future forecasts (no actuals or metrics)

| h | origin_date | target_date | target_date_status | price_origin | forecast | xgb_lo95 | xgb_hi95 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 2026-09-09 00:00:00 | 2026-09-10 00:00:00 | PROJECTED: weekday pattern, holidays/omissions unknown | 4401.550000 | 4446.587060 | 4361.772338 | 4478.077089 |
| 3 | 2026-09-09 00:00:00 | 2026-09-14 00:00:00 | PROJECTED: weekday pattern, holidays/omissions unknown | 4401.550000 | 4411.841577 | 4327.984542 | 4531.533348 |
| 5 | 2026-09-09 00:00:00 | 2026-09-16 00:00:00 | PROJECTED: weekday pattern, holidays/omissions unknown | 4401.550000 | 4424.730971 | 4355.106785 | 4482.541426 |
| 7 | 2026-09-09 00:00:00 | 2026-09-18 00:00:00 | PROJECTED: weekday pattern, holidays/omissions unknown | 4401.550000 | 4524.366277 | 4340.644675 | 4573.779236 |
| 14 | 2026-09-09 00:00:00 | 2026-09-29 00:00:00 | PROJECTED: weekday pattern, holidays/omissions unknown | 4401.550000 | 4320.463039 | 4323.558424 | 4521.473170 |
| 21 | 2026-09-09 00:00:00 | 2026-10-08 00:00:00 | PROJECTED: weekday pattern, holidays/omissions unknown | 4401.550000 | 4314.560887 | 4296.149291 | 4583.723567 |
| 30 | 2026-09-09 00:00:00 | 2026-10-21 00:00:00 | PROJECTED: weekday pattern, holidays/omissions unknown | 4401.550000 | 4504.552867 | 4315.807349 | 4783.526225 |



Future dates are projections from a declared source-weekday pattern, not confirmed exchange sessions. Holidays or missing future records can shift them.



## Checks and limitations

Process checks: 38 passed; 0 failed.

Train/Test diagnostics with no predeclared acceptance gate are REPORTED. Passing diagnostics does not establish forecast accuracy.

No separate validation estimate or validation gap exists. Train fitting loss is optimistic; fixed budgets do not establish absence of overfitting.

Test findings did not change parameters, epochs, transformations, seeds, or model selection.

These univariate models do not observe news, monetary policy, interest rates, inflation, or unexpected market events.

All rows, figures, summaries and fitted artifacts are under OUTPUT/XGBOOST_XAU/.

Tested Python executable: C:\Users\User\anaconda3\envs\vs_nine_tf\python.exe.



**Daily future forecasts**

Tables for 7 and 30 future steps are shown before metrics; dates follow the existing asset calendar convention.



**XGBoost prediction intervals**

xgb_lo95/xgb_hi95: nominal central 95% Train-only quantile intervals; no Test calibration or tuning. Fixed quantile rearrangement is audited. Empirical Test coverage is reported separately; 95% coverage is not guaranteed.

## Implementation reference
- Local reference: `../BTC_TEST/XGBOOST_BTC.ipynb` (read only; not rerun or modified).
- User protocol: `C:/Users/User/.codex/skills/time-series-forecasting/SKILL.md` and its asset-adaptation/model references.
- These notebooks adapt the local implementation; they do not claim replication of a separately tuned external study.